# 🧠 Personal Knowledge Base Chat

## What We're Building

A **personal second brain** — index your notes, bookmarks, and markdown files
into a searchable vector store and chat with them.

Think of it like having a conversation with everything you've ever written down.

## How It Works

```
Markdown notes ──┐
Text files ───────┼→ Load → Chunk → Embed → Local Vector Store
Bookmarks (URLs) ─┘                               ↓
                         User question → Search → Retrieve → LLM → Answer
```

## What Makes This Different

- **Multiple file types**: Markdown, plain text, HTML bookmarks
- **Persistent local store**: Data stays on your machine (ChromaDB on disk)
- **Incremental updates**: Add new notes without re-indexing everything
- **Conversation memory**: Follow-up questions reference chat history

## Stack
- **LangChain** — document loaders and orchestration
- **ChromaDB** — persistent local vector store
- **Ollama** — local LLM + embeddings

## Step 1 — Imports

In [ ]:
# !pip install langchain langchain-ollama langchain-community chromadb -q

from pathlib import Path
import hashlib

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document
from langchain_core.messages import HumanMessage, AIMessage

print("All imports successful!")

## Step 2 — Create Sample Knowledge Base Files

We'll create sample markdown notes, a text file, and a bookmark list
to simulate a personal knowledge base.

In [ ]:
# Create a sample knowledge base directory
kb_dir = Path("sample_knowledge_base")
kb_dir.mkdir(exist_ok=True)

# Note 1: Meeting notes
(kb_dir / "meeting_notes_jan.md").write_text("""
# Team Meeting — January 15, 2025

## Attendees
Sarah, Mike, Priya, James

## Key Decisions
- Migrate from PostgreSQL to CockroachDB for multi-region support
- Target completion: end of Q1
- James will lead the migration; Priya handles testing

## Action Items
- [ ] James: Create migration plan by Jan 20
- [ ] Priya: Set up staging environment by Jan 22
- [ ] Mike: Evaluate connection pooling options (PgBouncer vs built-in)

## Notes
- Current database handles 50K queries/sec, need 200K for expansion
- Budget approved: $45K/year for CockroachDB Enterprise
- Risk: OAuth service depends on PostgreSQL-specific features (LISTEN/NOTIFY)
""", encoding="utf-8")

# Note 2: Learning notes
(kb_dir / "rust_learning_notes.md").write_text("""
# Rust Learning Notes

## Ownership Rules
1. Each value has exactly one owner
2. When the owner goes out of scope, the value is dropped
3. You can have either ONE mutable reference OR any number of immutable references

## Key Concepts
- **Borrowing**: Passing references instead of ownership. Use `&` for immutable, `&mut` for mutable.
- **Lifetimes**: Annotations that tell the compiler how long references are valid. Syntax: `'a`
- **Pattern Matching**: `match` is exhaustive — must handle all cases. Use `_` for catch-all.
- **Result/Option**: No null! Use `Option<T>` for optional values, `Result<T, E>` for errors.

## Common Patterns
```rust
// Error handling with ?
fn read_file(path: &str) -> Result<String, io::Error> {
    let content = fs::read_to_string(path)?; // ? propagates error
    Ok(content)
}
```

## Resources
- The Rust Book: https://doc.rust-lang.org/book/
- Rustlings exercises: https://github.com/rust-lang/rustlings
""", encoding="utf-8")

# Note 3: Project ideas
(kb_dir / "project_ideas.txt").write_text("""
Project Ideas Brainstorm
========================

1. CLI tool for managing dotfiles across machines
   - Use Rust for speed, cross-platform
   - Git-backed sync with encryption for secrets
   - Similar to chezmoi but simpler

2. Local-first habit tracker
   - SQLite for storage, no cloud dependency
   - Terminal UI with ratatui (Rust) or textual (Python)
   - Weekly email digest using SMTP

3. Smart bookmark manager
   - Auto-tag bookmarks using LLM
   - Full-text search over saved pages
   - Export to markdown with categories

4. API health monitor
   - Ping endpoints every 5 minutes
   - Track response time trends
   - Alert via Telegram/Discord webhook
""", encoding="utf-8")

# Note 4: Bookmarks / links
(kb_dir / "bookmarks.md").write_text("""
# Saved Bookmarks

## Machine Learning
- [FastAI course](https://course.fast.ai/) — Best practical deep learning course
- [Papers With Code](https://paperswithcode.com/) — ML papers with implementations
- [Hugging Face](https://huggingface.co/) — Model hub and datasets

## System Design
- [System Design Primer](https://github.com/donnemartin/system-design-primer)
- [ByteByteGo](https://bytebytego.com/) — Alex Xu's system design newsletter

## Rust
- [Are We Web Yet](https://www.arewewebyet.org/) — Rust web ecosystem tracker
- [Awesome Rust](https://github.com/rust-unofficial/awesome-rust)

## Career
- Salary negotiation: Always negotiate. Start 15-20% above target.
- Interview prep: Do 2-3 LeetCode mediums per day for 4 weeks.
- Focus on system design for senior roles, not just algorithms.
""", encoding="utf-8")

# List what we created
files = list(kb_dir.iterdir())
print(f"Created {len(files)} knowledge base files:")
for f in files:
    size = f.stat().st_size
    print(f"  📄 {f.name} ({size} bytes)")

## Step 3 — Universal File Loader

A loader that handles multiple file types and adds useful metadata.

In [ ]:
def load_knowledge_base(directory: str | Path) -> list[Document]:
    """Load all supported files from a directory into Documents."""
    directory = Path(directory)
    supported = {".md", ".txt", ".text", ".markdown"}
    documents = []
    
    for file_path in sorted(directory.rglob("*")):
        if file_path.suffix.lower() not in supported:
            continue
        
        text = file_path.read_text(encoding="utf-8", errors="ignore")
        if len(text.strip()) < 10:
            continue
        
        # Create a content hash for deduplication
        content_hash = hashlib.md5(text.encode()).hexdigest()[:8]
        
        doc = Document(
            page_content=text,
            metadata={
                "source": file_path.name,
                "path": str(file_path),
                "type": file_path.suffix.lstrip("."),
                "size_bytes": file_path.stat().st_size,
                "content_hash": content_hash,
            },
        )
        documents.append(doc)
        print(f"  ✓ {file_path.name} ({len(text)} chars)")
    
    return documents


print("Loading knowledge base...")
raw_docs = load_knowledge_base(kb_dir)
print(f"\nLoaded {len(raw_docs)} documents")

## Step 4 — Chunk and Build Persistent Vector Store

We use **persistent storage** so the index survives restarts.
ChromaDB stores data to disk when you set `persist_directory`.

In [ ]:
# Split documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n## ", "\n# ", "\n\n", "\n", ". "],  # Markdown-aware
)
chunks = splitter.split_documents(raw_docs)
print(f"Split into {len(chunks)} chunks")

# Build persistent vector store
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")

PERSIST_DIR = "./kb_vectorstore"
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR,
    collection_name="personal_kb",
)

print(f"Vector store persisted to: {PERSIST_DIR}")
print(f"Total vectors: {vectorstore._collection.count()}")

## Step 5 — Build the Chat Interface with Memory

### Conversation Memory

Unlike simple Q&A, a personal KB chat should handle follow-ups:
```
User: "What database are we migrating to?"
Bot:  "CockroachDB for multi-region support."
User: "Why? And who's leading it?"          ← Needs context from previous turn
```

We maintain a **chat history** and include it in the prompt.

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0.3)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Prompt with chat history
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful personal knowledge assistant. You have access to the
user's notes, bookmarks, and documents. Answer questions based on this knowledge base.

Guidelines:
- Cite which file/note the information comes from
- If the knowledge base doesn't contain the answer, say so
- Be conversational and helpful
- Reference previous messages in the conversation when relevant

Knowledge base context:
{context}"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])


class KBChat:
    """Personal knowledge base chatbot with memory."""
    
    def __init__(self):
        self.history: list = []   # Chat history as LangChain messages
        self.chain = chat_prompt | llm | StrOutputParser()
    
    def ask(self, question: str) -> str:
        """Ask a question with conversation context."""
        # Retrieve relevant chunks
        docs = retriever.invoke(question)
        context = "\n\n".join(
            f"[Source: {d.metadata['source']}]\n{d.page_content}"
            for d in docs
        )
        
        # Generate answer
        answer = self.chain.invoke({
            "context": context,
            "chat_history": self.history,
            "question": question,
        })
        
        # Update history
        self.history.append(HumanMessage(content=question))
        self.history.append(AIMessage(content=answer))
        
        # Keep history manageable (last 10 turns)
        if len(self.history) > 20:
            self.history = self.history[-20:]
        
        return answer
    
    def clear_history(self):
        """Reset conversation history."""
        self.history.clear()
        print("🔄 History cleared")


chat = KBChat()
print("Knowledge base chat ready!")

## Step 6 — Chat with Your Knowledge Base!

In [ ]:
# First question
answer = chat.ask("What database migration is planned?")
print(f"💡 {answer}")

In [ ]:
# Follow-up — should use chat history
answer = chat.ask("Who is leading it and what's the budget?")
print(f"💡 {answer}")

In [ ]:
# Switch topic — search a different note
answer = chat.ask("What are the key Rust ownership rules I noted down?")
print(f"💡 {answer}")

In [ ]:
# Search bookmarks
answer = chat.ask("What are my saved resources for learning machine learning?")
print(f"💡 {answer}")

In [ ]:
# Search project ideas
answer = chat.ask("What project ideas involve Rust?")
print(f"💡 {answer}")

## Step 7 — Add New Notes Incrementally

Add new content to the knowledge base **without** re-indexing everything.

In [ ]:
def add_note(title: str, content: str, source: str = "quick_note") -> None:
    """Add a new note to the knowledge base."""
    doc = Document(
        page_content=f"# {title}\n\n{content}",
        metadata={"source": source, "type": "note"},
    )
    new_chunks = splitter.split_documents([doc])
    vectorstore.add_documents(new_chunks)
    print(f"✅ Added note '{title}' ({len(new_chunks)} chunks)")


# Add a new note
add_note(
    title="Docker Cheat Sheet",
    content="""## Common Commands
- `docker build -t myapp .` — Build image from Dockerfile
- `docker run -p 8080:80 myapp` — Run container with port mapping
- `docker compose up -d` — Start all services in background
- `docker exec -it container_name bash` — Shell into running container
- `docker logs -f container_name` — Follow container logs

## Tips
- Use multi-stage builds to reduce image size
- Never run as root inside containers
- Use .dockerignore to exclude node_modules, .git, etc.
""",
    source="docker_notes.md",
)

# Now search for it
answer = chat.ask("How do I shell into a running Docker container?")
print(f"💡 {answer}")

## Step 8 — Load Existing Store on Restart

In [ ]:
# On restart, load the existing vector store (no re-embedding needed)
def load_existing_kb(persist_dir: str = PERSIST_DIR):
    """Load an existing knowledge base from disk."""
    vs = Chroma(
        persist_directory=persist_dir,
        embedding_function=embeddings,
        collection_name="personal_kb",
    )
    count = vs._collection.count()
    print(f"Loaded KB from disk: {count} vectors")
    return vs


# Demo — normally you'd call this on notebook restart
# existing_vs = load_existing_kb()
print("Persistent storage available for future sessions")

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Multi-format loader** | Handles .md, .txt, .text files uniformly |
| **Persistent ChromaDB** | Saves vectors to disk, survives restarts |
| **Conversation memory** | Chat history enables follow-up questions |
| **MessagesPlaceholder** | LangChain's way to inject chat history |
| **Incremental indexing** | Add new docs without re-embedding existing ones |
| **Content hashing** | Detect duplicate documents on re-index |

## 🔧 Customization Ideas

- **PDF/HTML support**: Add more loaders for richer content types
- **Obsidian integration**: Load an Obsidian vault with wiki-links resolved
- **Auto-tagging**: Use LLM to tag incoming notes with topics
- **Scheduled re-index**: Watch a folder and re-index on file changes